<b><h1 style="font-size:30px;text-align: center;">- Podatkovno inženjerstvo - Vježba 5 -</h1></b>
<b><h1 style="font-size:25px;text-align: center;">- Nedostajuće vrijednosti i ekstremne vrijednosti (outlieri) -</h1></b>

<div style="text-align:center; margin:16px 0 28px;">
  <img
    src="https://futsal-dinamo.hr/wp-content/uploads/2021/01/Algebra-Bernays.png"
    alt="Algebra Bernays"
    style="width:600px; max-width:60%; height:auto; margin-top:25px;"
  >
</div>

---------------------

<b>*Izrađeno: listopad 2026.*</b>

<b>*Autor: Mislav Spajić, mag. ing. comp.*</b>

---------------------

# Kako raditi s ovom bilježnicom

> **Prvo spremi svoju kopiju:** `File → Save Notebook As…` → `EX_5_moj_rad.ipynb` i radi u njoj.
> Tako `git pull` nikad neće javiti sukob s tvojim izmjenama.

Svaka cjelina ima isti raspored:

| Oznaka | Što znači |
|---|---|
| 📖 **Teorija** | kratak podsjetnik; broj slajda odnosi se na **2. prezentaciju s predavanja** |
| ▶️ **Demo** | kod koji pokrećemo zajedno (`Shift + Enter`) |
| 🧩 **Zadatak** | ćelija s oznakom `# TODO` – zamijeni `...` svojim kodom |
| ✅ **Provjera** | pokreni odmah nakon zadatka; ispisat će ✅ ili objasniti što ne valja (provjere su u datoteci `provjere.py`) |
| ✏️ | odgovor upisuješ u markdown ćeliju (dvoklik na ćeliju, zatim `Shift + Enter`) |
| ⭐ | dodatni zadatak – ako ti ostane vremena na satu, ili za doma |

Ćelije pokreći **redom, odozgo prema dolje** – kasnije ćelije koriste varijable iz ranijih.
Ako nešto zapne: `Kernel → Restart Kernel and Run All Cells…` pa nastavi od mjesta gdje si stao/la.

# Sadržaj

| # | Cjelina | Podaci | ≈ min |
|---|---|---|---|
| 0 | Priprema okruženja | – | 5 |
| 1 | Kvaliteta podataka – brzi pregled | automobili | 7 |
| 2 | Otkrivanje nedostajućih vrijednosti | Titanic | 5 |
| 3 | Vrste nedostajanja: MCAR, MAR, MNAR | Titanic | 10 |
| 4 | Brisanje nedostajućih vrijednosti | Titanic | 6 |
| 5 | Imputacija (zamjena) nedostajućih vrijednosti | Titanic | 12 |
| 6 | Nedostajuće vrijednosti u vremenskim nizovima | kvaliteta zraka | 10 |
| 7 | Ekstremne vrijednosti – otkrivanje i obrada | niz s predavanja | 10 |
| 8 | Ekstremne vrijednosti na stvarnim podacima | automobili | 13 |
| 9 | ⭐ Završni zadatak: od sirovih do očišćenih podataka | automobili | 12 + doma |

**Skupovi podataka** (mapa `podaci/`):
* `titanic/` – putnici Titanica ([Kaggle: Titanic](https://www.kaggle.com/c/titanic))
* `air-quality-data-in-india/city_day.csv` – dnevna kvaliteta zraka u 26 indijskih gradova, 2015.–2020. ([Kaggle](https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india))
* `cars.csv` – tehnički podaci i cijene automobila ([Kaggle: Car Features and MSRP](https://www.kaggle.com/datasets/CooperUnion/cardataset))

Nazivi stupaca ostaju izvorni (engleski).

# 0. Priprema okruženja (≈ 5 min)

Radimo u Jupyter podu (`okruzenje/anaconda_jupyter.yaml`). Anaconda već ima `pandas`, `numpy`, `matplotlib`, `seaborn` i `scikit-learn`.
Doinstalirat ćemo samo **missingno** – biblioteku za crtanje nedostajućih vrijednosti.

In [ ]:
# Samo ako je nastavnik najavio ispravak materijala – povlači najnoviju verziju repozitorija.
# (Radiš u svojoj kopiji, pa se tvoje izmjene neće prebrisati.)
!git pull

In [ ]:
%pip install -q missingno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import missingno as msno
except ImportError:              # ako instalacija nije uspjela (npr. nema interneta), bilježnica radi i bez grafova missingno
    msno = None
    print("missingno nije dostupan – grafovi iz missingno bit će preskočeni.")

pd.set_option("display.max_columns", 30)
print("pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# Pomoćna funkcija koju koristimo kroz cijelu bilježnicu + provjere zadataka.

def tablica_nedostajucih(df):
    """Broj i postotak nedostajućih vrijednosti po stupcu (samo stupci koji ih imaju)."""
    broj = df.isna().sum()
    tablica = pd.DataFrame({"nedostaje": broj, "% redaka": (100 * broj / len(df)).round(1)})
    return tablica[tablica["nedostaje"] > 0].sort_values("nedostaje", ascending=False)

# Provjere zadataka nalaze se u datoteci provjere.py (u istoj mapi kao bilježnica).
# reload osigurava da se nakon git pull učita nova verzija bez ponovnog pokretanja kernela.
import importlib
import provjere
importlib.reload(provjere)

In [ ]:
titanic = pd.read_csv("podaci/titanic/train.csv")
titanic_test = pd.read_csv("podaci/titanic/test.csv")
zrak = pd.read_csv("podaci/air-quality-data-in-india/city_day.csv", parse_dates=["Date"])
auti = pd.read_csv("podaci/cars.csv")

for naziv, df in [("titanic", titanic), ("titanic_test", titanic_test), ("zrak", zrak), ("auti", auti)]:
    print(f"{naziv:13} {df.shape[0]:6} redaka × {df.shape[1]:2} stupaca")

# 1. Kvaliteta podataka – brzi pregled (≈ 7 min)

📖 **Teorija** (sl. 5–6, 21)

| Dimenzija | Pitanje |
|---|---|
| **Točnost** | Opisuje li podatak ispravno stvarni objekt ili događaj? |
| **Potpunost** | Postoje li sve vrijednosti potrebne za opis objekta? |
| **Dosljednost** | Je li isti podatak jednak u svim sustavima? |
| **Pravovremenost** | Je li podatak ažuran kad nam treba? |
| **Usklađenost** | Poštuje li dogovoreni tip, format i poslovna pravila? |
| **Cjelovitost (integritet)** | Mogu li se pratiti veze s povezanim podacima? |
| **Jedinstvenost** | Postoji li zapis samo jednom ili ima duplikata? |

Pravilo **1-10-100**: sprječavanje loših podataka stoji 1, ispravljanje 10, a neispravljena pogreška 100 jedinica.
Zato podatke pregledavamo **prije** nego ih upotrijebimo u izvještajima ili modelima.

▶️ **Demo** – prvi pogled na tablicu automobila. Ove tri naredbe pokrećemo nad svakom novom tablicom.

In [ ]:
auti.head()

In [ ]:
auti.info()          # tipovi stupaca i broj NE-praznih vrijednosti

In [ ]:
auti.describe()      # osnovna statistika numeričkih stupaca – pogledaj min i max!

In [ ]:
print("Potpuno istih redaka (duplikata):", auti.duplicated().sum())
tablica_nedostajucih(auti)

In [ ]:
# Najčešće cijene – nešto je sumnjivo
auti["MSRP"].value_counts().head(5)

🧩 **Zadatak 1** – cijena 2000 pojavljuje se više od tisuću puta. Je li to stvarna cijena?

In [ ]:
# TODO a) Koliko automobila ima cijenu (MSRP) točno 2000?
broj_cijena_2000 = ...

# TODO b) Koja je NAJNOVIJA godina proizvodnje (Year) među tim automobilima?
najnovija_godina_2000 = ...

print(broj_cijena_2000, najnovija_godina_2000)

In [ ]:
# ✅ Provjera
provjere.zadatak_1(broj_cijena_2000, najnovija_godina_2000)

✏️ **Zadatak 1 c)** Upiši koju dimenziju kvalitete narušava svaki problem (popis dimenzija je u tablici iznad).

| # | Problem u tablici `auti` | Narušena dimenzija |
|---|---|---|
| 1 | 715 potpuno istih redaka | |
| 2 | `Engine HP` nedostaje u 69 redaka | |
| 3 | mnogo automobila ima `MSRP = 2000` (rezultat zadatka 1 a–b) | |
| 4 | `highway MPG = 354` za benzinski Audi A6 (ostali benzinci imaju najviše 53) | |

# 2. Otkrivanje nedostajućih vrijednosti (≈ 5 min)

📖 **Teorija** (sl. 23–25) – nedostajuća vrijednost je vrijednost koja nije pohranjena za neku varijablu.
U pandasu se prikazuje kao `NaN` (ili `None`/`NaT`), a pronalazimo je s `isna()`.
Razlozi: oštećeni stari podaci, ljudska pogreška pri unosu, korisnik namjerno nije upisao vrijednost…

<center><img src="slike/Missin.png" width="700"/></center>

▶️ **Demo** – skup Titanic (stupac `Survived`: 1 = preživio, 0 = nije preživio).

In [ ]:
titanic.head()

In [ ]:
print("Skup za treniranje (train):")
display(tablica_nedostajucih(titanic))
print("Skup za testiranje (test):")
display(tablica_nedostajucih(titanic_test))

**missingno** prikazuje nedostajanje grafički:
* `msno.bar` – koliko je svaki stupac popunjen,
* `msno.matrix` – **gdje** su rupe; bijela crta = nedostajuća vrijednost u tom retku.

In [ ]:
if msno:
    msno.bar(titanic, figsize=(10, 4), fontsize=10)
    plt.show()

In [ ]:
if msno:
    msno.matrix(titanic, figsize=(10, 4), fontsize=10)
    plt.show()

# 3. Vrste nedostajanja: MCAR, MAR, MNAR (≈ 10 min)

📖 **Teorija** (sl. 26–32)

| Vrsta | Značenje | Primjer s predavanja |
|---|---|---|
| **MCAR** – potpuno slučajno | nedostajanje nema veze ni s jednom varijablom | upitnici izgubljeni u pošti |
| **MAR** – slučajno | nedostajanje se može objasniti **drugom, promatranom** varijablom | mlađi češće preskaču pitanje o potrošnji |
| **MNAR** – nesustavno | nedostajanje ovisi o **samoj nepoznatoj vrijednosti** | ljudi s visokim primanjima ne prijavljuju plaću |

**Kako to provjeriti u podacima?** Usporedimo udio nedostajućih vrijednosti po skupinama neke druge varijable.
Ako se udio jako razlikuje po skupinama, nedostajanje **nije MCAR** (vjerojatno je MAR).
MNAR iz samih podataka ne možemo dokazati, jer ne vidimo vrijednosti koje nedostaju – za to treba poznavati domenu.

▶️ **Demo** – nedostaje li dob (`Age`) jednako često u svim putničkim klasama (`Pclass`)?

In [ ]:
# isna() daje True/False; srednja vrijednost True/False stupca = udio True vrijednosti
age_po_klasi = titanic["Age"].isna().groupby(titanic["Pclass"]).mean()
print((age_po_klasi * 100).round(1).astype(str) + " %")

age_po_klasi.plot(kind="bar", title="Udio putnika bez podatka o dobi, po klasi", ylabel="udio", rot=0)
plt.show()

U 3. klasi dob nedostaje gotovo **5 puta češće** nego u 2. klasi (28 % prema 6 %).
Nedostajanje ovisi o promatranoj varijabli `Pclass`, dakle **nije MCAR – to je MAR**.

> Napomena: heatmap iz missingno (`msno.heatmap`) uspoređuje samo nedostajanje jednog stupca s nedostajanjem drugog.
> Ne pokazuje ovisi li nedostajanje o **vrijednostima** drugih stupaca, pa ovakav obrazac s njim ne bismo vidjeli.

🧩 **Zadatak 2**

In [ ]:
# TODO a) Udio nedostajućih vrijednosti stupca Cabin po klasi (Pclass) – isto kao u demu.
cabin_po_klasi = ...

# TODO b) Retci kojima nedostaje Embarked (luka ukrcaja).
embarked_nedostaje = ...

print(cabin_po_klasi)
embarked_nedostaje

In [ ]:
# ✅ Provjera
provjere.zadatak_2(cabin_po_klasi, embarked_nedostaje)

✏️ **Zadatak 2 c)** Za svaki stupac odaberi vrstu nedostajanja i obrazloži jednom rečenicom.

| Stupac | MCAR / MAR / MNAR | Obrazloženje |
|---|---|---|
| `Age` | MAR | nedostaje 28 % u 3. klasi, a 6 % u 2. klasi – ovisi o `Pclass` |
| `Cabin` | | |
| `Embarked` | | |

⭐ **Zadatak 2 d)** Ima li samo nedostajanje dobi veze s preživljavanjem?

In [ ]:
# TODO Prosjek stupca Survived grupiran po tome nedostaje li Age (True/False).
prezivljavanje_po_age = ...
prezivljavanje_po_age

In [ ]:
# ✅ Provjera
provjere.zadatak_2d(prezivljavanje_po_age)

# 4. Brisanje nedostajućih vrijednosti (≈ 6 min)

📖 **Teorija** (sl. 35, 38–43)

<center><img src="slike/Deletions.png" width="600"/></center>

* **Parcijalno brisanje (pairwise)** – izračuni jednostavno preskaču `NaN`; pandas to radi automatski (`mean`, `sum`…).
* **Brisanje redaka (listwise)** – `dropna()` uklanja cijeli redak u kojem nešto nedostaje.
* **Brisanje stupca** – `drop(columns=...)` kada stupac nema dovoljno podataka da bi bio koristan.

Brisanje je prihvatljivo kod **MCAR**-a i **malog** broja nedostajućih vrijednosti.
Kod MAR-a i MNAR-a brisanjem sustavno gubimo određenu skupinu (npr. putnike 3. klase), pa rezultat postaje pristran.

▶️ **Demo**

In [ ]:
# Pairwise: mean() preskače NaN – računa prosjek iz 714 poznatih vrijednosti, ne iz 891
print("Prosječna dob:", round(titanic["Age"].mean(), 2), "| iz", titanic["Age"].count(), "vrijednosti")

# Listwise bez razmišljanja:
print("Prije dropna():", titanic.shape, "→ poslije dropna():", titanic.dropna().shape)

🧩 **Zadatak 3**

In [ ]:
# TODO a) Obriši samo retke kojima nedostaje Age (pomoć: argument subset).
titanic_bez_age_nan = ...

# TODO b) Obriši cijeli stupac Cabin.
titanic_bez_cabin = ...

# TODO c) Koliki POSTOTAK redaka (0–100) gubimo s titanic.dropna() bez argumenata?
postotak_izgubljenih = ...

In [ ]:
# ✅ Provjera
provjere.zadatak_3(titanic_bez_age_nan, titanic_bez_cabin, postotak_izgubljenih)

# 5. Imputacija (zamjena) nedostajućih vrijednosti (≈ 12 min)

📖 **Teorija** (sl. 44–49, 58–66)

<center><img src="slike/Impute.png" width="550"/></center>

| Metoda | Za koje podatke | Napomena |
|---|---|---|
| konstanta (`0`, `"Nepoznato"`) | kad znamo što praznina znači | kod kategorija: „nedostaje” postaje zasebna kategorija |
| srednja vrijednost (*mean*) | numerički, bez ekstremnih vrijednosti | ekstremne vrijednosti je povlače |
| medijan | numerički, s ekstremnim vrijednostima | otporan na ekstremne vrijednosti |
| mod (najčešća vrijednost) | kategorijski | uvijek vraća postojeću kategoriju |
| **+ indikator nedostajanja** | uz bilo koju od gornjih | čuva informaciju da je vrijednost nedostajala |

Nedostaci jednostavne imputacije: sve praznine dobiju **istu** vrijednost, pa se distribucija iskrivljuje, a veze među stupcima se zanemaruju.
Postoji i rizik **curenja podataka** (*data leakage*): vrijednosti za popunjavanje smiju se izračunati samo iz skupa za treniranje.

▶️ **Demo** – osnovne metode u pandasu i u scikit-learnu (kao na slajdovima).

In [ ]:
print("Age – srednja vrijednost:", round(titanic["Age"].mean(), 2), "| medijan:", titanic["Age"].median())
print("Embarked – mod:", titanic["Embarked"].mode()[0])

titanic_jednostavno = titanic.copy()
titanic_jednostavno["Age"] = titanic_jednostavno["Age"].fillna(titanic_jednostavno["Age"].median())          # medijan
titanic_jednostavno["Embarked"] = titanic_jednostavno["Embarked"].fillna(titanic_jednostavno["Embarked"].mode()[0])  # mod
titanic_jednostavno["Cabin"] = titanic_jednostavno["Cabin"].fillna("Nepoznato")                              # konstanta

tablica_nedostajucih(titanic_jednostavno)      # prazna tablica = nema više nedostajućih vrijednosti

In [ ]:
# Isto pomoću scikit-learna (sl. 59–63)
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")              # ili "mean", "most_frequent", "constant"
age_sklearn = imputer.fit_transform(titanic[["Age"]])   # fit = izračunaj medijan, transform = popuni
print("Medijan koji je imputer naučio:", imputer.statistics_)

In [ ]:
# Koliko jedna ista vrijednost iskrivljuje distribuciju?
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
titanic["Age"].plot(kind="hist", bins=40, ax=ax[0], title="Age – izvorno (bez NaN)")
titanic_jednostavno["Age"].plot(kind="hist", bins=40, ax=ax[1], title="Age – nakon imputacije medijanom")
plt.show()

Svih 177 praznina dobilo je vrijednost 28, pa je nastao umjetni „toranj” na grafu.
Iz cjeline 3 znamo da nedostajanje dobi ovisi o klasi (MAR). Bolje je zato popunjavati **medijanom unutar svake klase**:
putnici 1. klase su u prosjeku stariji od putnika 3. klase.

🧩 **Zadatak 4** – imputacija medijanom po skupini i indikator nedostajanja.

In [ ]:
titanic_imp = titanic.copy()

# TODO 1) Indikator nedostajanja: stupac Age_nedostaje (True/False) – napravi ga PRIJE popunjavanja!
titanic_imp["Age_nedostaje"] = ...

# TODO 2) Medijan dobi za svaku klasu: Series s indeksom Pclass (1, 2, 3).
medijan_po_klasi = ...

# TODO 3) Popuni Age: svakom putniku medijan NJEGOVE klase.
#         Pomoć: titanic_imp["Pclass"].map(medijan_po_klasi) daje medijan klase za svaki redak.
titanic_imp["Age"] = ...

print(medijan_po_klasi)
titanic_imp[["Pclass", "Age", "Age_nedostaje"]].head(6)

In [ ]:
# ✅ Provjera
provjere.zadatak_4(titanic_imp, medijan_po_klasi)

⭐ **Zadatak 4 b)** – curenje podataka. Model se uči na `train` skupu, a primjenjuje na `test` skupu.
Vrijednosti za popunjavanje **moraju doći iz train skupa**, kao da test skup još ne postoji.

In [ ]:
titanic_test_imp = titanic_test.copy()

# TODO Popuni Age u test skupu medijanima klase IZRAČUNATIMA NA TRAIN SKUPU (varijabla medijan_po_klasi).
titanic_test_imp["Age"] = ...

In [ ]:
# ✅ Provjera
provjere.zadatak_4b(titanic_test, titanic_test_imp, medijan_po_klasi)

# 6. Nedostajuće vrijednosti u vremenskim nizovima (≈ 10 min)

📖 **Teorija** (sl. 36, 51–57) – u vremenskom nizu susjedni dani su slični, pa praznine popunjavamo iz susjednih vrijednosti:

| Metoda | pandas | Ideja |
|---|---|---|
| popunjavanje unaprijed (*forward fill*) | `.ffill()` | zadnja poznata vrijednost |
| popunjavanje unatrag (*backward fill*) | `.bfill()` | sljedeća poznata vrijednost |
| linearna interpolacija | `.interpolate()` | pravac između dviju poznatih vrijednosti – dobro za nizove s trendom |

> ⚠️ Na slajdovima piše `fillna(method='ffill')`. Taj oblik je zastario u pandasu 2.1, a u **pandasu 3 više ne radi**.
> Pišite `.ffill()` i `.bfill()`.

▶️ **Demo** – mali niz sa slajda 52:

In [ ]:
test = pd.Series(range(6), dtype="float")
test.loc[2:4] = np.nan
pd.DataFrame({"izvorno": test, "ffill": test.ffill(), "bfill": test.bfill(), "interpolate": test.interpolate()})

▶️ **Demo** – kvaliteta zraka. Tablica sadrži **26 gradova jedan ispod drugoga**, poredanih po gradu pa po datumu.

In [ ]:
stupci = zrak.select_dtypes("number").columns.tolist()     # mjerenja: PM2.5, PM10, NO, …, AQI
print(zrak["City"].nunique(), "gradova,", zrak["Date"].min().date(), "–", zrak["Date"].max().date())
tablica_nedostajucih(zrak)

In [ ]:
# Ahmedabad, veljača–ožujak 2015.: tri dana bez mjerenja Xylene – usporedba metoda
ahm = zrak[zrak["City"] == "Ahmedabad"].set_index("Date")["Xylene"].iloc[50:65]
pd.DataFrame({"izvorno": ahm, "ffill": ahm.ffill(), "bfill": ahm.bfill(), "interpolate": ahm.interpolate()})

▶️ **Demo** – greška koja se često vidi: `ffill()` nad **cijelom** tablicom.

In [ ]:
zrak_naivno = zrak.copy()
zrak_naivno[stupci] = zrak_naivno[stupci].ffill()

granica = zrak.index[zrak["City"] == "Aizawl"][0]          # prvi redak grada Aizawl
print("PRIJE:")
display(zrak.loc[granica - 2: granica + 2, ["City", "Date", "PM2.5", "Xylene", "AQI"]])
print("POSLIJE naivnog ffill():")
display(zrak_naivno.loc[granica - 2: granica + 2, ["City", "Date", "PM2.5", "Xylene", "AQI"]])

Aizawl **nikad nije mjerio** Xylene, a naivni `ffill()` mu je za svaki dan upisao **1,87** – zadnju vrijednost iz Ahmedabada.
Zato se popunjavanje mora raditi **unutar svakog grada zasebno**.

🧩 **Zadatak 5** – ispravi popunjavanje unaprijed tako da vrijednosti ne prelaze iz jednog grada u drugi.

In [ ]:
zrak_ffill = zrak.copy()

# TODO ffill() po gradu: grupiraj po 'City', odaberi stupce `stupci` i pozovi ffill().
zrak_ffill[stupci] = ...

In [ ]:
# ✅ Provjera
provjere.zadatak_5(zrak, zrak_ffill, zrak_naivno, stupci)

⭐ **Zadatak 5 b)** Mumbai nema AQI čak **1222 dana zaredom**. Ima li smisla „izmisliti” tri godine podataka?
Interpoliraj AQI po gradu, ali popuni najviše **7 uzastopnih dana** i samo **između** dviju poznatih vrijednosti.
Pomoć: `s.interpolate(limit=7, limit_area="inside")` i `groupby(...)[...].transform(...)`.

In [ ]:
# TODO Interpolirani AQI po gradu (najviše 7 dana, samo unutar poznatih vrijednosti).
aqi_interp = ...

In [ ]:
# ✅ Provjera
provjere.zadatak_5b(zrak, aqi_interp)

# 7. Ekstremne vrijednosti – otkrivanje i obrada (≈ 10 min)

📖 **Teorija** (sl. 69–88) – ekstremna vrijednost (*outlier*) znatno odstupa od ostalih opažanja.
Može biti **stvarna** (vrlo skup automobil) ili **pogreška** (tipfeler pri unosu). Prvo je zato pitanje *zašto* je vrijednost ekstremna.

* Na **srednju vrijednost** jako utječu ekstremne vrijednosti, a **medijan** je na njih otporan (sl. 70).
* **IQR metoda:** `IQR = Q3 − Q1`; ekstremno je sve ispod `Q1 − 1,5·IQR` ili iznad `Q3 + 1,5·IQR` (sl. 77). To je ista granica kao „brkovi” na kutijastom dijagramu (*boxplot*).
* **Z-vrijednost:** `z = (x − srednja vrijednost) / std`; ekstremno je ako je `|z| > 3` (sl. 80–81).
* **Obrada** (sl. 83–88): uklanjanje (*trimming*), ograničavanje (*capping/flooring*), zamjena medijanom.

▶️ **Demo** – niz s predavanja i radnog lista 6.1:

In [ ]:
sample = np.array([15, 101, 18, 7, 13, 16, 11, 21, 5, 15, 10, 9])
print("srednja vrijednost:", sample.mean().round(2), "| medijan:", np.median(sample))

plt.figure(figsize=(7, 1.8))
plt.boxplot(sample, vert=False)
plt.title("Otkrivanje ekstremnih vrijednosti kutijastim dijagramom")
plt.show()

🧩 **Zadatak 6** – napiši dvije funkcije. Rezultati moraju odgovarati tvom ručnom izračunu iz radnog lista 6.1.

In [ ]:
def iqr_granice(podaci, k=1.5):
    """Vraća (donja, gornja) granicu po IQR metodi."""
    # TODO Izračunaj q1 i q3 (np.percentile), zatim IQR i obje granice.
    ...
    return ...


def z_vrijednosti(podaci):
    """Vraća niz Z-vrijednosti za svaki element."""
    # TODO (x - srednja vrijednost) / standardna devijacija – koristi np.mean i np.std
    return ...

In [ ]:
# ✅ Provjera (vrijednosti iz rješenja radnog lista 6.1)
provjere.zadatak_6(iqr_granice, z_vrijednosti, sample)

▶️ **Demo** – tri načina obrade (radni list 6.2):

In [ ]:
q1_s, q3_s = np.percentile(sample, [25, 75])
gornja_s = q3_s + 1.5 * (q3_s - q1_s)                                 # IQR gornja granica
p10, p90 = np.percentile(sample, [10, 90])

uklonjeno = sample[sample <= gornja_s]                                   # trimming
ograniceno = np.clip(sample, p10, p90)                                 # capping/flooring na 10. i 90. percentil
medijan_umjesto = np.where(sample > gornja_s, np.median(sample), sample) # zamjena medijanom

pd.DataFrame({
    "verzija": ["izvorno", "uklanjanje", "capping 10./90.", "zamjena medijanom"],
    "broj vrijednosti": [len(sample), len(uklonjeno), len(ograniceno), len(medijan_umjesto)],
    "srednja vrijednost": [sample.mean(), uklonjeno.mean(), ograniceno.mean(), medijan_umjesto.mean()],
}).round(2)

Primijeti: capping na 10. i 90. percentil promijenio je i vrijednosti **5** i **21**, koje nisu ekstremne.
Blaža varijanta je ograničavanje na IQR granice iz zadatka 6: `np.clip(sample, donja, gornja)`.
Z-vrijednost na malim uzorcima loše radi, jer sam outlier jako povećava standardnu devijaciju. IQR je tu pouzdaniji.

# 8. Ekstremne vrijednosti na stvarnim podacima (≈ 13 min)

▶️ **Demo** – cijene automobila (`MSRP`, u USD). Najprije uklanjamo duplikate.

In [ ]:
auti_bd = auti.drop_duplicates()
print("Bez duplikata:", auti_bd.shape)

fig, ax = plt.subplots(1, 2, figsize=(12, 2.8))
sns.boxplot(x=auti_bd["MSRP"], ax=ax[0]).set_title("MSRP")
sns.boxplot(x=auti_bd["MSRP"], ax=ax[1], log_scale=True).set_title("MSRP – logaritamska skala")
plt.show()

Na običnoj skali „ekstremnih” je jako mnogo, jer su cijene **desno asimetrične** (puno jeftinih, malo jako skupih automobila).
Na logaritamskoj skali slika je jasnija, a lijevo se vidi i gusta nakupina oko 2000 – to su naše lažne „cijene” iz zadatka 1.

▶️ **Demo** – pristup iz mnogih online tutorijala: IQR na **svim** numeričkim stupcima odjednom i brisanje svakog retka koji igdje „strši”.

In [ ]:
numericki = auti_bd.select_dtypes("number")
q1, q3 = numericki.quantile(0.25), numericki.quantile(0.75)
iqr = q3 - q1
izvan = (numericki < q1 - 1.5 * iqr) | (numericki > q3 + 1.5 * iqr)

auti_slijepo = auti_bd[~izvan.any(axis=1)]
uklonjeni = auti_bd[izvan.any(axis=1)]

print(f"Uklonjeno {len(uklonjeni)} od {len(auti_bd)} redaka ({100 * len(uklonjeni) / len(auti_bd):.0f} %)")
pd.DataFrame({"donja granica": q1 - 1.5 * iqr, "gornja granica": q3 + 1.5 * iqr, "redaka izvan": izvan.sum()})

Uklonjena je gotovo četvrtina tablice. **Što je zapravo obrisano?**

🧩 **Zadatak 7** – istraži tablicu `uklonjeni`.

In [ ]:
# TODO a) Koliko je Fordova (Make == "Ford") uklonjeno, a koliko ih ukupno ima u auti_bd?
fordovi_uklonjeno = ...
fordovi_ukupno = ...

# TODO b) Koliko je električnih automobila (Engine Fuel Type == "electric") uklonjeno, a koliko ih ukupno ima?
elektricni_uklonjeno = ...
elektricni_ukupno = ...

# TODO c) Koliko je automobila s MSRP == 2000 OSTALO u auti_slijepo (IQR ih nije prepoznao)?
cijena_2000_ostalo = ...

print("Ford:", fordovi_uklonjeno, "/", fordovi_ukupno)
print("Električni:", elektricni_uklonjeno, "/", elektricni_ukupno)
print("MSRP = 2000 ostalo:", cijena_2000_ostalo)
uklonjeni.nlargest(5, "MSRP")[["Make", "Model", "Year", "MSRP"]]

In [ ]:
# ✅ Provjera
provjere.zadatak_7(fordovi_uklonjeno, fordovi_ukupno, elektricni_uklonjeno, elektricni_ukupno, cijena_2000_ostalo)

✏️ **Zadatak 7 d)** Objasni rezultate. Pomoć: pogledaj tablicu granica iznad i stupce `Popularity`, `Engine Cylinders` i `highway MPG` kod električnih automobila.

| Slučaj | Zašto ga je IQR označio (ili propustio)? | Pogreška ili stvarna vrijednost? | Što napraviti? |
|---|---|---|---|
| svi Fordovi | | | |
| svi električni automobili | | | |
| Bugatti Veyron (2 milijuna USD) | | | |
| `MSRP = 2000` | | | |
| Audi A6, `highway MPG = 354` | | | |

▶️ **Demo** – kad su ekstremne vrijednosti stvarne, a smetaju nam (npr. kod prosjeka), umjesto brisanja možemo ih **ograničiti**:

In [ ]:
g_msrp = (q3 + 1.5 * iqr)["MSRP"]          # gornja IQR granica za MSRP (iz prethodnog dema)
pd.DataFrame({
    "verzija": ["izvorno", "uklanjanje (IQR)", "capping na IQR granice"],
    "redaka": [len(auti_bd), (auti_bd["MSRP"] <= g_msrp).sum(), len(auti_bd)],
    "srednja vrijednost MSRP": [auti_bd["MSRP"].mean(),
                                auti_bd.loc[auti_bd["MSRP"] <= g_msrp, "MSRP"].mean(),
                                auti_bd["MSRP"].clip(upper=g_msrp).mean()],
    "medijan MSRP": [auti_bd["MSRP"].median(),
                     auti_bd.loc[auti_bd["MSRP"] <= g_msrp, "MSRP"].median(),
                     auti_bd["MSRP"].clip(upper=g_msrp).median()],
}).round(0)

Medijan se gotovo ne mijenja, a srednja vrijednost se mijenja jako. Ako nas zanima „tipična” cijena, medijan je dovoljan i ništa ne treba brisati.

# 9. ⭐ Završni zadatak: od sirovih do očišćenih podataka (≈ 12 min + doma)

Na vježbi 4 crtali smo lakehouse sa zonama **Bronze → Silver → Gold**.
Ovdje pišemo upravo korak **Bronze → Silver** za tablicu automobila: funkciju koja prima sirovu tablicu i vraća očišćenu kopiju.
Za svaku odluku mora postojati obrazloženje, a odluke se bilježe u dnevnik ispod funkcije.

Na satu kreni s koracima 1–3; ostatak dovrši doma. Ćelija s provjerom za svaki korak zasebno ispisuje ✅ ili ❌.

In [ ]:
def ocisti_aute(df):
    """Bronze -> Silver: vraća očišćenu KOPIJU tablice automobila (ulaznu tablicu ne mijenja)."""
    df = df.copy()

    # TODO 1) Ukloni potpune duplikate.
    ...

    # TODO 2) MSRP == 2000 znači "cijena nepoznata":
    #         - dodaj stupac MSRP_nepoznat (True/False),
    #         - zamijeni 2000 s NaN.
    ...

    # TODO 3) highway MPG: kod automobila koji NISU električni vrijednosti iznad 60 postavi na NaN.
    #         (Električni s ~100 MPGe su ispravni – njih ne diraj!)
    ...

    # TODO 4) Engine Cylinders: električni automobili nemaju cilindre -> upiši 0 gdje nedostaje.
    #         (Mazda RX-7/RX-8 imaju Wankelov motor bez klasičnih cilindara – ostavi NaN.)
    ...

    # TODO 5) Engine HP:
    #         - dodaj indikator Engine_HP_nedostaje,
    #         - popuni medijanom iste marke (Make), npr. groupby("Make")["Engine HP"].transform("median"),
    #         - što ostane prazno (marka nema nijednu poznatu snagu), popuni medijanom cijelog stupca.
    ...

    # TODO 6) Skupi automobili (Bugatti, Lamborghini…) su STVARNI – ne briši ih.
    #         Dodaj stupac log_MSRP = np.log10(MSRP) za analize kojima smetaju ekstremne vrijednosti.
    ...

    return df


silver = ocisti_aute(auti)
silver.head()

In [ ]:
# ✅ Provjera – svaki korak zasebno
provjere.zadatak_9(auti, silver)

✏️ **Dnevnik odluka** – za svaki korak napiši zašto je odluka dobra i koji je njezin rizik ili nedostatak.

| Korak | Stupac | Odluka | Zašto? | Rizik / nedostatak |
|---|---|---|---|---|
| 1 | svi | brisanje duplikata | | |
| 2 | `MSRP` | 2000 → NaN + indikator | | |
| 3 | `highway MPG` | > 60 (osim električnih) → NaN | | |
| 4 | `Engine Cylinders` | električni → 0 | | |
| 5 | `Engine HP` | medijan po marki + indikator | | |
| 6 | `MSRP` | ekstremne cijene zadržane, dodan `log_MSRP` | | |

⭐⭐ **Za one koji žele više:**
* `Engine Fuel Type` nedostaje za tri Suzuki Verone iz 2004. – popuni vrijednošću iz ostalih redaka istog modela.
* `Number of Doors` nedostaje za Teslu Model S – koliko vrata ima taj automobil? (Znanje o domeni je također izvor podataka.)
* Je li medijan snage po marki dobar izbor za električne automobile kojima nedostaje `Engine HP`? Predloži bolju skupinu.

# Sažetak

* Prije svake analize pokreni **`info()`, `describe()`, `isna().sum()` i `duplicated().sum()`**. Pogledaj i najčešće vrijednosti (`value_counts`), jer nedostajuće vrijednosti mogu biti skrivene iza „normalnog” broja.
* **Vrsta nedostajanja** (MCAR / MAR / MNAR) određuje metodu: provjeri udio nedostajućih vrijednosti po skupinama drugih stupaca.
* **Brisanje** samo za malo nedostajućih vrijednosti i MCAR. **Imputacija** po skupini je bolja od jedne vrijednosti za sve, a **indikator** čuva informaciju o nedostajanju.
* Vrijednosti za imputaciju računaj samo na **train** skupu (bez curenja podataka).
* Vremenske nizove popunjavaj **unutar svake serije** (`groupby`), a dugačke rupe ne izmišljaj (`limit`).
* IQR i Z-vrijednost samo **predlažu** ekstremne vrijednosti. Za svaku odluči je li pogreška ili stvarna vrijednost: pogrešku ispravi ili postavi na `NaN`, stvarnu zadrži, ograniči ili koristi medijan ili logaritamsku skalu.

**Izvori i dodatno čitanje**
* 2. prezentacija s predavanja: *Rad s nedostajućim podacima* i radni listovi.
* Dio primjera i slike u mapi `slike/` prilagođeni su iz Kaggle bilježnice *A Guide to Handling Missing values in Python* (Parul Pandey).
* [pandas – Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html) · [scikit-learn – Imputation](https://scikit-learn.org/stable/modules/impute.html) · [missingno](https://github.com/ResidentMario/missingno)